# ЛР № 4. Витрина городского велопроката Сеула

**Основной вопрос:** Как различается спрос на аренду велосипедов по сезонам и часам суток?

Продолжаем [Project Brief](../docs/project_brief.md) и
[решения ЛР № 3](../docs/data_quality_report.md). Готовим данные для основного вопроса; полный расчёт метрик и содержательные выводы остаются следующему
этапу. 

Срезы: Сезон аренды, час суток.

Метрики: Среднее число аренд по часу, доля аренд по сезону.

Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md).

## 1. Параметры и вход

Логическая дата берётся из пути подготовленного `staging`, а не из даты запуска.
Записи об аренде относятся к периоду `2017-12-01` – `2018-11-30`. `source_ingested_on=2026-10-15`
обозначает загрузку исходных данных и не используется как время события.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром
проектного `.venv`. Стенд должен работать; настройки S3 читаются из `.env`.
Ingestion и ЛР № 3 повторно выполнять не нужно.

Справочник уже загружен на стенд в `raw`, отдельного объекта справочника в `staging`
нет. Читаем существующий CSV-снимок за `2026-10-15` напрямую из S3, проверяем его
и включаем названия в широкую таблицу.


In [16]:
from datetime import date

DATASET_SLUG = "seoul_bike"
SOURCE_INGESTED_ON = "2026-10-15"
MART_NAME ="rent"
date.fromisoformat(SOURCE_INGESTED_ON)

PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_PATH = f"s3://staging/{PREFIX}/staging.parquet"
MART_KEY = f"{PREFIX}/{MART_NAME}.parquet"
MART_PATH = f"s3://mart/{MART_KEY}"

print(f"Вход:  {STAGING_PATH}")
print(f"Выход: {MART_PATH}")

Вход:  s3://staging/seoul_bike/source_ingested_on=2026-10-15/staging.parquet
Выход: s3://mart/seoul_bike/source_ingested_on=2026-10-15/rent.parquet


In [17]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing_settings)}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402


def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

check_results = []

def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"

connection = duckdb.connect()
configure_s3(connection)
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
source_before = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")


In [18]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
source_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "Date", "Hour", "Seasons", "Rented Bike Count", "Functioning Day", "quality_functioning_day_consistency_violated",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

source_counts = connection.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (
            WHERE "Rented Bike Count" IS NULL
        ) AS null_rent_counts,
        count(*) FILTER (
            WHERE quality_functioning_day_consistency_violated
        ) AS flagged_rows,
        count(*) FILTER (
            WHERE "Date" IS NULL
               OR "Hour" IS NULL
               OR "Hour" NOT BETWEEN 0 AND 23
               OR "Seasons" IS NULL
               OR "Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
               OR "Rented Bike Count" IS NULL
               OR "Rented Bike Count" < 0
               OR "Functioning Day" IS NULL
               OR quality_functioning_day_consistency_violated IS NULL
        ) AS critical_violations
    FROM staging
""").fetchone()
staging_rows, source_null_rent_counts, source_flagged_rows, source_critical = source_counts
check("Критические нарушения ЛР № 3 на входе", source_critical, 0)
check("Пропуски числа аренд", source_null_rent_counts, 0)

show_table(
    ["Строк staging", "Пропуски числа аренд", "Строк с флагом", "Критические нарушения"],
    [source_counts],
)
show_table(
    ["Нужное поле staging", "Тип"],
    [(name, source_schema[name]) for name in sorted(required_fields)],
)


rental_profile = connection.execute("""
    SELECT
        min("Rented Bike Count"),
        max("Rented Bike Count"),
        count(*) FILTER (WHERE NOT "Functioning Day"),
        count(*) FILTER (
            WHERE quality_functioning_day_consistency_violated
        )
    FROM staging
""").fetchone()

show_table(
    ["Минимум аренд", "Максимум аренд", "Нерабочих часов", "Часов с флагом"],
    [rental_profile],
)


Строк staging,Пропуски числа аренд,Строк с флагом,Критические нарушения
8760,0,0,0


Нужное поле staging,Тип
Date,DATE
Functioning Day,BOOLEAN
Hour,BIGINT
Rented Bike Count,BIGINT
Seasons,VARCHAR
quality_functioning_day_consistency_violated,BOOLEAN


Минимум аренд,Максимум аренд,Нерабочих часов,Часов с флагом
0,3556,295,0



Ограничение: при расчёте среднего спроса учитывать только рабочие часы.

## 2. Гранулярность, ключ и меры

**Одна строка таблицы `rent` представляет один час работы системы Seoul Bike Sharing
в определённую календарную дату** из выбранного снимка `staging`. 
Сезон и час суток — срезы этой часовой записи.

**Бизнес-ключ — `(Date, Hour)`.** Он идентифицирует час наблюдения системы. 
В ЛР № 3
проверены заполненность обеих частей и отсутствие повторов: 8 760 строк,
0 пропусков ключа и 0 повторов. Полных дубликатов также нет. В витрине не создаётся технический идентификатор.

Сохраняем все строки `staging`. Число строк (`count(*)`) означает число
часовых наблюдений, а не число аренд. Для основного вопроса используются
сезон (`Seasons`) и час (`Hour`), включая их сочетание.

| Мера | Единица и использование | Допустимая агрегация |
| --- | --- | --- |
| `Rented Bike Count` | Число арендованных велосипедов за час; основная мера спроса | Суммируется по непересекающимся часам для итога аренд. Среднее считается как `avg` по подходящим часовым строкам; средние групп нельзя складывать, а при объединении нужно учитывать число наблюдений. |
| Число строк (`count(*)`) | Число часовых наблюдений; знаменатель и контроль покрытия, не число аренд | Суммируется по непересекающимся группам часов. |

Для доли аренд сезона числитель — сумма `Rented Bike Count` по сезону,
знаменатель — сумма этой меры за весь анализируемый набор; долю нельзя
получить из количества строк. Сезонные суммы по непересекающимся сезонам
можно складывать. Витрина сама не хранит сезонные итоги или средние: они
рассчитываются последующей агрегацией часовых записей.

`Functioning Day = false` означает недоступность сервиса; соответствующие
нули аренд не трактуются как наблюдение отсутствия спроса при расчёте среднего.
Их нужно исключать из знаменателя среднего спроса либо показывать отдельно,
сохраняя при этом записи в витрине и учитывая их в покрытии. В ЛР № 3 таких
часов 295; нарушений согласованности и выставленных quality-флагов — 0.

Дата наблюдения (`Date`) относится к периоду `2017-12-01`–`2018-11-30`.
Логическая дата `2026-10-15` в пути — дата загрузки исходного снимка, а не
время события. Набор покрывает один полный год, поэтому сезонные сравнения
описывают только этот период и не гарантируют повторения в другие годы.


## 3. SQL-преобразование

Формируем широкую почасовую таблицу `rent` из полей, выбранных для вопроса о сезонном
и суточном профиле спроса. Явно перечисляем все поля, приводим имена к `snake_case`,
значения и уровень детализации не меняем. Сохраняем признак рабочего дня и флаг
проверки качества из ЛР № 3. Фильтрации, агрегации, соединений и дедупликации нет.

### Сборка широкой таблицы

In [21]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE rent AS
    SELECT
        "Date" AS date,
        "Hour" AS hour,
        "Seasons" AS seasons,
        "Rented Bike Count" AS rented_bike_count,
        "Functioning Day" AS functioning_day,
        quality_functioning_day_consistency_violated AS functioning_day_violation
    FROM staging
""")
prepared_rows = connection.execute("SELECT count(*) FROM rent").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
prepared_flagged_rows = connection.execute("""
    SELECT count(*)
    FROM rent
    WHERE functioning_day_violation
""").fetchone()[0]

check("Число строк после преобразования", prepared_rows, staging_rows)
check("Исключённые строки", excluded_rows, 0)
check("Перенос quality-флага", prepared_flagged_rows, source_flagged_rows)

show_table(
    ["Поле витрины", "Тип"],
    [row[:2] for row in connection.execute("DESCRIBE rent").fetchall()],
)
show_table(
    ["Дата", "Час", "Сезон", "Аренды", "Рабочий день", "Нарушение-рабочего-дня"],
    connection.execute("""
        SELECT date, hour, seasons, rented_bike_count,
               functioning_day, functioning_day_violation
        FROM rent
        ORDER BY date, hour
        LIMIT 5
    """).fetchall(),
)

Поле витрины,Тип
date,DATE
hour,BIGINT
seasons,VARCHAR
rented_bike_count,BIGINT
functioning_day,BOOLEAN
functioning_day_violation,BOOLEAN


Дата,Час,Сезон,Аренды,Рабочий день,Нарушение-рабочего-дня
2017-12-01,0,Winter,254,True,False
2017-12-01,1,Winter,204,True,False
2017-12-01,2,Winter,173,True,False
2017-12-01,3,Winter,107,True,False
2017-12-01,4,Winter,78,True,False


## 4. Запись и повторное чтение S3

Записываем `rent` в Parquet по пути `MART_PATH`. Повторное выполнение заменяет
только целевой объект, не затрагивая другие объекты S3. Все строки `staging`
сохраняются, поэтому входное, подготовленное и повторно прочитанное количества
должны совпасть.

После записи открываем новое соединение DuckDB и читаем именно объект из бакета
`mart`; автоматическое добавление полей из каталогов отключено.

In [22]:
assert MART_PATH == f"s3://mart/{PREFIX}/rent.parquet"
assert MART_KEY == f"{PREFIX}/rent.parquet"
client.head_bucket(Bucket="mart")
client.delete_object(Bucket="mart", Key=MART_KEY)
connection.execute(f"""
    COPY rent TO {literal(MART_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")

published = duckdb.connect()
configure_s3(published)
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_rent AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
mart_rows = published.execute("SELECT count(*) FROM published_rent").fetchone()[0]
mart_flagged_rows = published.execute("""
    SELECT count(*) FROM published_rent WHERE functioning_day_violation
""").fetchone()[0]
check("Количество строк после чтения из S3", mart_rows, staging_rows)
check("Флаги качества после чтения из S3", mart_flagged_rows, source_flagged_rows)

output_head = client.head_object(Bucket="mart", Key=MART_KEY)
show_table(
    ["Объект", "Строк", "Строк с quality-флагом", "Размер, байт"],
    [(MART_PATH, mart_rows, mart_flagged_rows, output_head["ContentLength"])],
)

Объект,Строк,Строк с quality-флагом,"Размер, байт"
s3://mart/seoul_bike/source_ingested_on=2026-10-15/rent.parquet,8760,0,32521


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Все запросы к `published_trips` ниже читают записанный Parquet на S3.
Ожидания следуют из модели и входного `staging`. При нарушении любого условия
`assert` останавливает выполнение. Сводка в конце показывает фактическое
и ожидаемое значение каждой проверки.

### Количество и схема

Все строки `staging` сохраняются. Проверяем нулевое число исключений, баланс
`staging = mart + исключено`, совпадение с подготовленной таблицей
и схему записанного файла.

In [8]:
mart_rows = published.execute("SELECT count(*) FROM published_trips").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
check("Исключённые строки", excluded_rows, 0)
check("Баланс входных и сохранённых строк", staging_rows, mart_rows + excluded_rows)
check("Количество после чтения S3", mart_rows, prepared_rows)

expected_schema = [
    ("pickup_at", "TIMESTAMP"),
    ("dropoff_at", "TIMESTAMP"),
    ("pickup_zone_id", "INTEGER"),
    ("pickup_zone_name", "VARCHAR"),
    ("pickup_borough", "VARCHAR"),
    ("dropoff_zone_id", "INTEGER"),
    ("dropoff_zone_name", "VARCHAR"),
    ("dropoff_borough", "VARCHAR"),
    ("pickup_date", "DATE"),
    ("pickup_hour", "BIGINT"),
    ("pickup_isodow", "BIGINT"),
    ("duration_minutes", "DOUBLE"),
    ("trip_distance", "DOUBLE"),
    ("total_amount", "DOUBLE"),
    ("negative_total_flag", "BOOLEAN"),
]
actual_schema = [row[:2] for row in published.execute("DESCRIBE published_trips").fetchall()]
check("Имена, порядок и типы пятнадцати полей", actual_schema, expected_schema)
show_table(["Staging", "Подготовлено", "Прочитано из mart", "Исключено"], [
    (staging_rows, prepared_rows, mart_rows, excluded_rows)
])
show_table(["Поле", "Тип"], actual_schema)

Staging,Подготовлено,Прочитано из mart,Исключено
48235,48235,48235,0


Поле,Тип
pickup_at,TIMESTAMP
dropoff_at,TIMESTAMP
pickup_zone_id,INTEGER
pickup_zone_name,VARCHAR
pickup_borough,VARCHAR
dropoff_zone_id,INTEGER
dropoff_zone_name,VARCHAR
dropoff_borough,VARCHAR
pickup_date,DATE
pickup_hour,BIGINT


### Обязательные поля и меры

Ключа поездки нет, поэтому проверка его уникальности неприменима. Проверяем
заполненность времён, кодов, названий и районов обеих зон, календарных признаков,
длительности и флага. Без них теряется смысл строки или необходимого среза.
Для `total_amount` и `trip_distance` допускаем исходные пропуски: новые
появляться не должны, их количество сравниваем со `staging`.

Длительность должна оставаться положительной по решению ЛР № 3. Верхнюю границу
в ЛР № 3 не задавали: максимум показываем для контроля и не вводим скрытого
отсечения. Отрицательный минимум суммы допустим при сохранённом флаге.

In [10]:
null_counts = published.execute("""
    SELECT
        count(*) FILTER (WHERE pickup_at IS NULL),
        count(*) FILTER (WHERE dropoff_at IS NULL),
        count(*) FILTER (WHERE pickup_zone_id IS NULL),
        count(*) FILTER (WHERE pickup_zone_name IS NULL),
        count(*) FILTER (WHERE pickup_borough IS NULL),
        count(*) FILTER (WHERE dropoff_zone_id IS NULL),
        count(*) FILTER (WHERE dropoff_zone_name IS NULL),
        count(*) FILTER (WHERE dropoff_borough IS NULL),
        count(*) FILTER (WHERE pickup_date IS NULL),
        count(*) FILTER (WHERE pickup_hour IS NULL),
        count(*) FILTER (WHERE pickup_isodow IS NULL),
        count(*) FILTER (WHERE duration_minutes IS NULL),
        count(*) FILTER (WHERE negative_total_flag IS NULL),
        count(*) FILTER (WHERE trip_distance IS NULL),
        count(*) FILTER (WHERE total_amount IS NULL)
    FROM published_trips
""").fetchone()
required_names = [
    name for name, _ in expected_schema if name not in {"trip_distance", "total_amount"}
]
for name, null_count in zip(required_names, null_counts[:-2], strict=True):
    check(f"NULL в обязательном поле {name}", null_count, 0)
check("NULL расстояния относительно staging", null_counts[-2], source_distance_profile[0])
check("NULL суммы относительно staging", null_counts[-1], source_null_amount)
show_table(["Поле", "NULL", "Ожидается"], [
    *[(name, value, 0) for name, value in zip(required_names, null_counts[:-2], strict=True)],
    ("trip_distance", null_counts[-2], source_distance_profile[0]),
    ("total_amount", null_counts[-1], source_null_amount),
])

measure_profile = published.execute("""
    SELECT min(duration_minutes), max(duration_minutes),
           min(total_amount), max(total_amount),
           count(*) FILTER (WHERE duration_minutes <= 0)
    FROM published_trips
""").fetchone()
source_amount_range = connection.execute("""
    SELECT min(total_amount), max(total_amount) FROM staging
""").fetchone()
check("Неположительные длительности", measure_profile[4], 0)
check("Минимум и максимум суммы сохранены", measure_profile[2:4], source_amount_range)
show_table(["Мера", "Минимум", "Максимум", "Единица"], [
    ("duration_minutes", measure_profile[0], measure_profile[1], "Минуты"),
    ("total_amount", measure_profile[2], measure_profile[3], "USD"),
])

Поле,NULL,Ожидается
pickup_at,0,0
dropoff_at,0,0
pickup_zone_id,0,0
pickup_zone_name,0,0
pickup_borough,0,0
dropoff_zone_id,0,0
dropoff_zone_name,0,0
dropoff_borough,0,0
pickup_date,0,0
pickup_hour,0,0


Мера,Минимум,Максимум,Единица
duration_minutes,0.016666666666666666,1439.65,Минуты
total_amount,-114.0,371.4,USD


### Календарные признаки, длительность и флаг

Проверяем диапазоны часа и ISO-дня, соответствие признаков временным меткам,
положительную длительность и январский период посадки. Сверяем флаг с самим
отрицательным значением и количество помеченных записей со `staging`.

In [12]:
derived_violations = published.execute("""
    SELECT count(*) FROM published_trips
    WHERE pickup_hour NOT BETWEEN 0 AND 23
       OR pickup_isodow NOT BETWEEN 1 AND 7
       OR pickup_date IS DISTINCT FROM CAST(pickup_at AS DATE)
       OR pickup_hour IS DISTINCT FROM EXTRACT(hour FROM pickup_at)
       OR pickup_isodow IS DISTINCT FROM EXTRACT(isodow FROM pickup_at)
       OR duration_minutes IS DISTINCT FROM date_diff('second', pickup_at, dropoff_at) / 60.0
       OR pickup_at < TIMESTAMP '2025-01-01'
       OR pickup_at >= TIMESTAMP '2025-02-01'
""").fetchone()[0]
flag_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE negative_total_flag) AS flagged_rows,
        count(*) FILTER (WHERE total_amount < 0) AS negative_rows,
        count(*) FILTER (
            WHERE negative_total_flag IS DISTINCT FROM coalesce(total_amount < 0, false)
        ) AS flag_mismatches
    FROM published_trips
""").fetchone()
check("Нарушения вычисленных полей и периода", derived_violations, 0)
check("Количество флагов относительно staging", flag_profile[0], source_flagged_rows)
check("Отрицательные суммы имеют флаг", flag_profile[1], source_flagged_rows)
check("Несоответствия суммы и флага", flag_profile[2], 0)
show_table(["Нарушения признаков", "Флаг TRUE", "Отрицательная сумма", "Несоответствия"], [
    (derived_violations, *flag_profile)
])

Нарушения признаков,Флаг TRUE,Отрицательная сумма,Несоответствия
0,152,152,0


### Сохранение значений и кратности строк

Бизнес-ключа поездки нет. Сверяем все пятнадцать полей записанной витрины
с ожидаемой проекцией всех строк `staging`, обогащённых справочником, через
`EXCEPT ALL` в обе стороны. `EXCEPT ALL` учитывает число совпадающих строк:
совпавшие аналитические проекции не должны исчезать. Проверяются также точный
перенос отрицательных сумм с флагами и расстояние, а также названия и районы обеих зон.

In [13]:
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW expected_trips AS
    SELECT
        s.lpep_pickup_datetime AS pickup_at,
        s.lpep_dropoff_datetime AS dropoff_at,
        s.PULocationID AS pickup_zone_id,
        pz.Zone AS pickup_zone_name,
        pz.Borough AS pickup_borough,
        s.DOLocationID AS dropoff_zone_id,
        dz.Zone AS dropoff_zone_name,
        dz.Borough AS dropoff_borough,
        CAST(s.lpep_pickup_datetime AS DATE) AS pickup_date,
        EXTRACT(hour FROM s.lpep_pickup_datetime) AS pickup_hour,
        EXTRACT(isodow FROM s.lpep_pickup_datetime) AS pickup_isodow,
        date_diff('second', s.lpep_pickup_datetime, s.lpep_dropoff_datetime)
            / 60.0 AS duration_minutes,
        s.trip_distance,
        s.total_amount,
        s.quality_negative_total_amount_is_correction_violated AS negative_total_flag
    FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false) AS s
    LEFT JOIN published_zones AS pz ON s.PULocationID = pz.LocationID
    LEFT JOIN published_zones AS dz ON s.DOLocationID = dz.LocationID
""")
content_difference = published.execute("""
    SELECT 'Отсутствуют в mart' AS direction, count(*) AS differing_rows
    FROM (SELECT * FROM expected_trips EXCEPT ALL SELECT * FROM published_trips)
    UNION ALL
    SELECT 'Лишние в mart', count(*)
    FROM (SELECT * FROM published_trips EXCEPT ALL SELECT * FROM expected_trips)
""").fetchall()
for direction, differing_rows in content_difference:
    check(direction, differing_rows, 0)
show_table(["Направление сверки", "Различающихся строк"], content_difference)

Направление сверки,Различающихся строк
Отсутствуют в mart,0
Лишние в mart,0


### Пригодность для вопросов проекта

Один контрольный запрос группирует записи по коду и названию зоны посадки,
а также району. Показываем первые десять групп; баланс считаем по всем группам.
Сумма их чисел записей должна равняться общему числу строк. Запрос подтверждает
возможность работы с географическим срезом после обогащения; полный расчёт
медиан, средних и содержательные выводы остаются следующему этапу.

Для первого уточняющего вопроса проверены наличие и точный перенос расстояния,
длительности и суммы с правилами их будущего использования. Для второго - коды,
названия и районы зон посадки и высадки, включая покрытие обоих соединений.
Группировка пар зон и анализ связи мер остаются следующему этапу.


In [14]:
zone_counts = published.execute("""
    SELECT pickup_zone_id, pickup_zone_name, pickup_borough, count(*) AS records_count
    FROM published_trips
    GROUP BY pickup_zone_id, pickup_zone_name, pickup_borough
    ORDER BY records_count DESC, pickup_zone_id
""").fetchall()
check("Баланс контрольной группировки", sum(row[3] for row in zone_counts), mart_rows)
show_table(["Код зоны", "Зона посадки", "Район", "Число записей"], zone_counts[:10])
show_table(["Всего групп", "Сумма записей по всем группам"], [
    (len(zone_counts), sum(row[3] for row in zone_counts))
])

Код зоны,Зона посадки,Район,Число записей
74,East Harlem North,Manhattan,11672
75,East Harlem South,Manhattan,8060
43,Central Park,Manhattan,2776
95,Forest Hills,Queens,2236
166,Morningside Heights,Manhattan,2126
97,Fort Greene,Brooklyn,1993
41,Central Harlem,Manhattan,1974
82,Elmhurst,Queens,1798
65,Downtown Brooklyn/MetroTech,Brooklyn,1508
130,Jamaica,Queens,1324


Всего групп,Сумма записей по всем группам
212,48235


### Контрольная сводка

Дополнительно проверяем, что метаданные обоих входных объектов не изменились за время
выполнения. Все проверки относятся к конкретному снимку, а не доказывают
полноту или достоверность сведений о реальных поездках.

In [15]:
source_after = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")
check("Входной объект не изменён", (
    source_after["ETag"], source_after["ContentLength"], source_after["LastModified"]
), (
    source_before["ETag"], source_before["ContentLength"], source_before["LastModified"]
))
zones_after = client.head_object(Bucket="raw", Key=ZONES_KEY)
check("Объект справочника не изменён", (
    zones_after["ETag"], zones_after["ContentLength"], zones_after["LastModified"]
), (
    zones_before["ETag"], zones_before["ContentLength"], zones_before["LastModified"]
))
show_table(["Проверка", "Получено", "Ожидается", "Итог"], [
    (name, actual, expected, "Пройдена" if passed else "Нарушение")
    for name, actual, expected, passed in check_results
])

# Контрольная сводка выполненного ноутбука.
run_summary = {
    "staging_rows": staging_rows,
    "prepared_rows": prepared_rows,
    "mart_rows": mart_rows,
    "excluded_rows": excluded_rows,
    "schema": actual_schema,
    "null_counts": null_counts,
    "measure_profile": measure_profile,
    "distance_profile": distance_profile,
    "source_distance_profile": source_distance_profile,
    "distance_policy_profile": distance_policy_profile,
    "derived_violations": derived_violations,
    "flag_profile": flag_profile,
    "content_difference": content_difference,
    "lookup_profile": lookup_profile,
    "join_profile": join_profile,
    "dropoff_join_profile": dropoff_join_profile,
    "saved_zone_profile": saved_zone_profile,
    "saved_dropoff_zone_profile": saved_dropoff_zone_profile,
    "zone_counts": zone_counts,
    "checks_passed": sum(passed for _, _, _, passed in check_results),
    "checks_total": len(check_results),
    "source_etag": source_after["ETag"],
    "source_size": source_after["ContentLength"],
    "zones_etag": zones_after["ETag"],
    "zones_size": zones_after["ContentLength"],
}
show_table(["Проверок", "Пройдено", "Строк mart", "Сохранено флагов"], [
    (len(check_results), run_summary["checks_passed"], mart_rows, flag_profile[0])
])
published.close()
connection.close()

Проверка,Получено,Ожидается,Итог
Наличие нужных полей staging,set(),set(),Пройдена
Критические нарушения ЛР № 3 на входе,0,0,Пройдена
Наличие полей справочника,set(),set(),Пройдена
Пропуски ключа справочника,0,0,Пройдена
Повторяющиеся ключи справочника,0,0,Пройдена
Пустые названия зон и районов,0,0,Пройдена
Количество строк после JOIN зоны посадки,48235,48235,Пройдена
Посадки без соответствия до записи,0,0,Пройдена
Количество строк после JOIN зоны высадки,48235,48235,Пройдена
Высадки без соответствия до записи,0,0,Пройдена


Проверок,Пройдено,Строк mart,Сохранено флагов
50,50,48235,152


<a id="result"></a>
## 6. Оформите результат

Модель, происхождение всех выходных полей, краткий итог проверок и ограничения
описаны в [docs/mart.md](../docs/mart.md). Демонстрационная витрина поддерживает
основной и два уточняющих вопроса проекта. Для студенческой работы достаточно
одного выбранного вопроса из принятого `Project Brief`.

Итог сборки: все 48 235 строк `staging` сохранены в витрине, исключённых строк нет.
Гранулярность и кратность строк сохранены; оба соединения со справочником
не изменили количество записей. В выходном файле 15 полей, сохранены 152 флага
отрицательной суммы. Все 50 проверок сборки пройдены.

Витрина сохранена и прочитана из S3:

```text
s3://mart/green_tripdata/source_ingested_on=2026-09-30/trips.parquet
```

Для дальнейшего анализа зафиксировано общее правило: 2 629 строк с нулевым
расстоянием и 21 строка с расстоянием больше 1000 остаются для остальных
показателей, но исключаются из анализа связи расстояния с длительностью
и стоимостью. При сравнении со стоимостью также требуется заполненная
неотрицательная сумма. Это условие будущего расчёта, а не фильтр сборки витрины.

Ограничения: бизнес-ключа поездки нет; соответствие справочнику не доказывает
точность географии, а его снимок не описывает историю изменения зон.
Отрицательный флаг обозначает знак суммы, причина неизвестна. Длительности
от одной секунды до почти суток сохранены по правилам ЛР № 3 и требуют
осторожности при интерпретации. Полный расчёт метрик и содержательные выводы
остаются следующему этапу.

Ноутбук сохранён с актуальными SQL-запросами и результатами выполнения.
Подробные проверки и таблицы результатов находятся в [разделе 5](#checks);
в описании модели оставлен краткий итог.